# Reproducible Colab notebook — `sbc`

### Explainable Deep Ensemble Learning for Multi-Scale Streamflow Bias Correction in GloFAS-ERA5 across Snow-Influenced Transboundary Basins of Central Asia

Open-source code artifact accompanying the MDPI *Water* manuscript. `sbc` learns and removes the
systematic errors of the **GloFAS-ERA5** global river-discharge reanalysis in the snow- and
glacier-fed headwaters of Central Asia, at both the operational **decadal** (10-day) scale
(primary, ~60 gauges across 7 core basins) and the **daily** scale (Naryn case study), and
explains every correction with SHAP attribution tied to snow processes.

**This notebook has two paths:**

| Path | Credentials | Downloads | Run-all default |
|------|-------------|-----------|-----------------|
| **A — Self-contained synthetic reproduction** | none | none | runs end-to-end |
| **B — Real-data pipeline** (optional) | your *own* free Copernicus CDS/EWDS token | CA-discharge (24 MB) + GloFAS (multi-GB) + ERA5-Land | gated off (`RUN_REAL = False`) |

Path A reproduces the whole framework — log-residual learning, the baseline ladder, the
gradient-boosting trio, the EA-LSTM, the regime-aware probabilistic flagship and the stacked
ensemble, scored across the temporal / leave-one-basin-out / prediction-in-ungauged-regions
(PUR) validation matrix, plus SHAP explainability — on a **physically-grounded synthetic
benchmark with a *known* injected GloFAS-style bias**. It needs no account and no data
download, so `Runtime > Run all` completes start-to-finish on a free Colab CPU runtime
(a GPU runtime speeds up the two deep models but is optional).

> **Secrets.** Copernicus credentials are *never* committed. `configs/secrets.env` is
> git-ignored; you supply your **own** ECMWF Personal Access Token at runtime (Section 4),
> entered through a hidden `getpass` prompt so it is never printed or stored in the notebook.

_Grant BR24993128 (Science Committee, MES RK). Datasets: GloFAS-ERA5, ERA5-Land, CA-discharge
(Zenodo 8147591), HydroATLAS — all openly available and cited in the README._

## 1 · Get the code

Point the notebook at the `sbc` repository. By default it shallow-clones the published
repository into `/content`; edit `REPO_URL` to the real URL. If you instead mounted the
project from Google Drive or uploaded it, set `SBC_ROOT` to that folder and the clone is
skipped automatically.

`sbc.config` resolves the project root from the `SBC_ROOT` environment variable, so the
identical package code runs unchanged on Colab, Linux and Windows.

In [ ]:
import os, sys, subprocess, pathlib

# --- edit this to the published repository URL ------------------------------
REPO_URL = "https://github.com/your-org/sbc.git"
# Where the repo lives on this runtime. If you mounted it from Drive, set e.g.
# SBC_ROOT = "/content/drive/MyDrive/MDPI-Q1-2026" before running this cell.
SBC_ROOT = os.environ.get("SBC_ROOT", "/content/MDPI-Q1-2026")

if not pathlib.Path(SBC_ROOT, "src", "sbc").exists():
    print("cloning", REPO_URL, "->", SBC_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, SBC_ROOT], check=True)
else:
    print("using existing checkout at", SBC_ROOT)

os.environ["SBC_ROOT"] = SBC_ROOT
sys.path.insert(0, os.path.join(SBC_ROOT, "src"))   # import sbc without installing it
print("SBC_ROOT =", SBC_ROOT)

## 2 · Install dependencies

The self-contained synthetic reproduction (Path A) needs only the core scientific + ML
stack; **PyTorch ships pre-installed on Colab**, so the pin below is satisfied without a
reinstall. The heavier geospatial + Copernicus stack (`geopandas`, `cdsapi`, `netcdf4`,
`rioxarray`, `xarray`, `dask`, ...) is only required for the optional real-data pipeline and
is installed later, on demand, from `requirements.txt`.

In [ ]:
%pip install -q "numpy>=1.24" "pandas>=2.0" "scipy>=1.10" "scikit-learn>=1.3" \
    "torch>=2.1" "xgboost>=2.0" "lightgbm>=4.0" "catboost>=1.2" \
    "shap>=0.44" "optuna>=3.4" "properscoring>=0.1" \
    "pyyaml>=6.0" "pyarrow>=14.0" "matplotlib>=3.7" "tqdm>=4.65"

## 3 · Verify the environment

In [ ]:
import sbc
from sbc.config import PATHS
import torch

print("sbc          ", sbc.__version__)
print("project root ", PATHS.root)
print("torch        ", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 4 · Copernicus credentials — *optional* (Path B only)

**Path A (the synthetic reproduction below) needs no credentials — you can skip this cell.**

The real-data pipeline streams GloFAS-ERA5 from the **EWDS** portal and ERA5-Land from the
**CDS** portal. You must use your **own** free ECMWF account and Personal Access Token, and
accept the licence for `cems-glofas-historical` (EWDS) and
`reanalysis-era5-land-monthly-means` (CDS) on each portal's dataset page first. A single
token works for both portals; only the endpoint URL differs.

The cell below is **off by default**. When enabled it reads the token through a hidden
`getpass` prompt (never printed, never hard-coded) and writes the git-ignored
`configs/secrets.env`. **Do not commit that file.**

In [ ]:
import getpass
from pathlib import Path

WRITE_CREDENTIALS = False   # set True only to run the optional real-data pipeline (Path B)

if WRITE_CREDENTIALS:
    token = getpass.getpass("ECMWF Personal Access Token (input hidden): ").strip()
    secrets = Path(os.environ["SBC_ROOT"]) / "configs" / "secrets.env"
    secrets.parent.mkdir(parents=True, exist_ok=True)
    secrets.write_text(
        "CDS_URL=https://cds.climate.copernicus.eu/api\n"
        f"CDS_KEY={token}\n"
        "EWDS_URL=https://ewds.climate.copernicus.eu/api\n"
        f"EWDS_KEY={token}\n")
    print("wrote", secrets, "(git-ignored - never commit this file)")
else:
    print("Skipped - the synthetic reproduction below needs no credentials.")

---
# Part A · Self-contained synthetic reproduction

Everything below runs end-to-end with **no credentials and no downloads.**

## 5 · Build the synthetic modelling table

`sbc.synthetic.generate` simulates a nival-glacial water balance per gauge (winter snow
store, degree-day spring freshet, summer glacier melt, soil-moisture baseflow) and then
injects the GloFAS-ERA5 error signatures documented for snow-dominated catchments — a
systematic negative volume bias, an early-shifted/damped melt freshet and compressed
variability. Because the injected bias is *known*, this controlled benchmark verifies, before
any real data is touched, that the framework recovers it. The output is a tidy modelling
table with the exact schema of the assembled real-data table (`code`, `basin`, `domain`,
`scale`, `date`, `q_obs`, `q_glofas`, + forcing + static attributes).

In [ ]:
from sbc.synthetic import generate
from sbc import schemas
from sbc.validation.metrics import evaluate

df = generate(scale="decadal", seed=1234)     # decadal = the operational planning scale
df = schemas.validate(df)                       # guarantees the log_residual ML target

print(f"{len(df):,} rows | {df.code.nunique()} gauges | {df.basin.nunique()} basins "
      f"| domains={sorted(df.domain.unique())}")
print("raw GloFAS median KGE' ~=", round(evaluate(df.q_obs, df.q_glofas)["kge"], 3),
      "(this is the skill we aim to improve)")
df[["code", "basin", "domain", "date", "q_obs", "q_glofas", "log_residual"]].head()

## 6 · Feature engineering & hydrological-regime classification

Features (lagged/rolling snow, melt, temperature and precipitation descriptors, plus static
catchment attributes) are discovered at run time from the table — never hard-coded — and each
row is assigned a hydrological regime (accumulation, melt-freshet, glacier-melt,
rain-on-snow, recession) used later for regime-conditional explainability.

In [ ]:
from sbc.features.engineering import build_features
from sbc.features.regimes import classify_regimes

feat = classify_regimes(build_features(df, scale="decadal")).reset_index(drop=True)
print("engineered model-input features:", len(schemas.feature_columns(feat)))
print("hydrological regimes           :", sorted(feat.regime.unique()))
feat[["date", "regime", "swe", "smlt", "t2m_mean", "log_residual"]].head()

## 7 · Run the experiment across the leakage-safe validation matrix

`run_experiment` is the package's config-driven entry point. It evaluates the model fleet
across three protocols — per-gauge **temporal** holdout, **leave-one-basin-out (LOBO)** and
**prediction-in-ungauged-regions (PUR)**, the last being the strongest generalisation test —
and reports per-gauge skill of the corrected discharge against both the observations and the
raw GloFAS benchmark.

The publication run uses `configs/default.yaml` unchanged (`quick=False`, full Optuna
tuning, the full 62-gauge synthetic table). For a fast, GPU-optional Colab run we (i) pass
`quick=True` (fewer epochs, no per-fold HPO) and (ii) override the synthetic table to a
smaller size so the two deep models train quickly on CPU. **Delete the override and set
`quick=False` to reproduce at publication scale.**

In [ ]:
import yaml, tempfile
from pathlib import Path
from sbc.experiment import load_config, run_experiment

cfg = load_config()                                  # configs/default.yaml - source of truth
# Colab-speed override (remove for the publication-scale synthetic run):
cfg["data"]["synthetic"] = {"n_basins": 4, "gauges_per_basin": [2, 3],
                            "years": 8, "seed": 1234}
cfg_path = Path(tempfile.gettempdir()) / "sbc_colab.yaml"
cfg_path.write_text(yaml.safe_dump(cfg))

# Full fleet: baseline ladder + boosting + EA-LSTM + the probabilistic flagship + stacking.
# For a ~2 min CPU run, drop the deep/stacked members:
#   FLEET = ["scaling", "qmap", "lgbm", "catboost"]
FLEET = ["scaling", "qmap", "lgbm", "catboost", "ealstm", "regimeprobnet", "stacked"]

out = run_experiment(source="synthetic", scale="decadal", quick=True,
                     config=str(cfg_path), models=FLEET, do_shap=False)
results, summary = out["results"], out["summary"]
summary

## 8 · Skill before vs after correction

Per-gauge KGE′ of the corrected discharge by model and split; the dashed red line is the raw
GloFAS-ERA5 median. The learned correctors lift skill well above the uncorrected reanalysis,
including under PUR (training on the core domain, testing on the held-out transfer basins).

In [ ]:
import matplotlib.pyplot as plt

splits = [s for s in ["temporal", "lobo", "pur"] if s in set(results.split)]
fig, axes = plt.subplots(1, len(splits), figsize=(5.2 * len(splits), 4.2), squeeze=False)
for ax, split in zip(axes[0], splits):
    sub = results[results.split == split]
    order = sub.groupby("model")["kge"].median().sort_values().index
    ax.boxplot([sub[sub.model == m]["kge"].dropna() for m in order],
               labels=list(order), vert=False)
    ax.axvline(sub["kge_raw"].median(), color="r", ls="--", label="raw GloFAS")
    ax.set_title(f"KGE' - {split}"); ax.set_xlim(-0.5, 1.0)
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

## 9 · Explainability — SHAP attribution onto snow drivers

Fit a LightGBM corrector on the temporal-train split and attribute its predicted log-residual
with TreeSHAP on the held-out test split. The global importance and the beeswarm show the
correction loading physically onto snow-water-equivalent, snowmelt and temperature features —
reported as physical consistency within the model, not proven causation.

In [ ]:
from sbc.models.boosting import LightGBMCorrector
from sbc.validation.splits import temporal_split
from sbc.explain import shap_analysis as X

tr, te = temporal_split(feat)
lgbm = LightGBMCorrector(n_optuna_trials=0).fit(feat[tr])
shap_res = X.tree_shap(lgbm, feat[te], max_samples=2000)
gi = X.global_importance(shap_res)
print("snow-process features among the top 15 drivers:",
      X.snow_features(list(gi.feature.head(15))))
gi.head(12)

In [ ]:
from IPython.display import Image

png = X.save_beeswarm(shap_res,
                      Path(os.environ["SBC_ROOT"]) / "results" / "shap" / "beeswarm_synthetic.png",
                      title="SHAP attribution of the GloFAS log-residual correction (synthetic)")
Image(str(png))

---
# Part B · Real-data pipeline (optional)

This part reproduces the **actual experiment** on GloFAS-ERA5, ERA5-Land and the CA-discharge
observations. It is **gated off by default** (`RUN_REAL = False`) so `Run all` stays
self-contained. To enable it you must, in order:

1. write your **own** Copernicus credentials in **Section 4** (`WRITE_CREDENTIALS = True`),
2. accept the dataset licences on the CDS and EWDS portals,
3. set `RUN_REAL = True` below, then run the remaining cells.

Downloads are large and slow: CA-discharge is ~24 MB (Zenodo), the GloFAS-ERA5 v4.0 daily
discharge pull is multi-GB (~2 h, resumable), and ERA5-Land monthly forcing adds more.
Use a Colab runtime with ample disk, or mount Google Drive so downloads persist across
sessions.

### B.1 · Install the geospatial + Copernicus stack and download the data

In [ ]:
RUN_REAL = False   # set True only after Sections 4 (credentials) + licence acceptance

if RUN_REAL:
    # heavier deps needed only here: geopandas, pyogrio, cdsapi, netcdf4, rioxarray, ...
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(Path(SBC_ROOT) / "requirements.txt")], check=True)

    import urllib.request
    from sbc.config import PATHS
    PATHS.ca_discharge.mkdir(parents=True, exist_ok=True)
    gpkg = PATHS.ca_discharge / "CA-discharge.gpkg"
    if not gpkg.exists():
        urllib.request.urlretrieve(
            "https://zenodo.org/records/8147591/files/CA-discharge.gpkg?download=1", gpkg)
    print("CA-discharge:", gpkg.stat().st_size // 1_000_000, "MB")

    from sbc.data.glofas import download_glofas, download_uparea
    from sbc.data.era5_land import download_era5_land
    download_uparea()      # small upstream-area grid for gauge -> river-pixel snapping
    download_glofas()      # GloFAS-ERA5 v4.0 daily discharge (multi-GB, ~2 h, resumable)
    download_era5_land()   # ERA5-Land monthly snow/meteorological forcing
else:
    print("RUN_REAL is False - skipping the real-data download.")

### B.2 · Assemble the real modelling table and run the experiment

Extract observed series + static attributes from the GeoPackage, snap each gauge to its
GloFAS river pixel, aggregate ERA5-Land to the contributing basins, assemble the modelling
table and run the same fleet across the validation matrix. Swap `scale="daily"` for the
high-resolution Naryn case study.

In [ ]:
if RUN_REAL:
    from sbc.data import ca_discharge, glofas, era5_land
    ca_discharge.main()      # processed/{gauges, discharge_*, static_attributes}.parquet
    glofas.main_extract()    # interim/glofas_*_at_gauges.parquet
    era5_land.main_extract() # interim/era5land_*_at_basins.parquet

    real = run_experiment(source="real", scale="decadal", quick=True,
                          models=["scaling", "qmap", "lgbm", "catboost",
                                  "ealstm", "regimeprobnet", "stacked"])
    display(real["summary"])
else:
    print("RUN_REAL is False - skipping the real-data experiment.")

---
## Notes & reproducibility

* **Artifacts.** Every run writes under `results/` — per-gauge and summary skill tables
  (`results/tables/`, Parquet + CSV mirror), a JSON report, and SHAP outputs
  (`results/shap/`). On Colab, mount Google Drive and set `SBC_ROOT` to a Drive folder to
  keep them.
* **Single source of truth.** All settings live in `configs/default.yaml`; the only Colab
  deviations are `quick=True` and the smaller synthetic table in Section 7. Reproduce at
  publication scale by removing both.
* **Validation honesty.** Splits are leakage-safe by construction (`sbc.validation.splits`):
  temporal holdout never trains on a gauge's future; LOBO and PUR keep whole basins/domains
  out of training. PUR (core Syr-Darya/Chu/Talas → held-out Amu Darya) is the strongest test.
* **Secrets.** `configs/secrets.env` is git-ignored and must never be committed; provide your
  own Copernicus token at runtime. The synthetic path needs none.
* **Citation.** Please cite the accompanying MDPI paper and the underlying datasets
  (GloFAS-ERA5, ERA5-Land, CA-discharge — Zenodo 8147591, HydroATLAS) as listed in the README.